In [36]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

tickers = ['NVDA', 'AAPL', 'GOOG', 'MSFT', 'AMZN',
           'META', 'AVGO', 'TSLA', 'WMT', 'LLY']
portfolio = yf.download(tickers, start="2020-01-01", end="2025-01-01")['Close']
portfolio.dropna(inplace=True)

[*********************100%***********************]  10 of 10 completed


In [37]:
def getPortReturn(weights, df):
    returns = df.pct_change(1).dropna()
    num_stocks = len(returns.columns)
    exp_ret_portfolio = np.dot(np.transpose(weights),
                              returns.mean()) * 250
    return exp_ret_portfolio

In [38]:
def getPortRisk(weights, df):
    returns = df.pct_change(1).dropna()
    num_stocks = len(returns.columns)
    vcv = returns.cov(ddof=1)

    var_p = np.dot(np.transpose(weights),
               np.dot(vcv, weights))
    sd_p = np.sqrt(var_p)
    sd_p_annual = sd_p * np.sqrt(250)

    return sd_p_annual

In [39]:
num_stocks = len(portfolio.columns)
init_weights = [1 / num_stocks] * num_stocks

In [40]:
print("Initial annual returns portfolio:",
      getPortReturn(init_weights, portfolio))
print("Initial annual risk portfolio:",
      getPortRisk(init_weights, portfolio))

Initial annual returns portfolio: 0.3975010146231638
Initial annual risk portfolio: 0.2814241802561516


In [57]:
ticker = '^GSPC'
market = yf.download(ticker, start="2020-01-01", end="2025-01-01")['Close']
market.dropna(inplace=True)

[*********************100%***********************]  1 of 1 completed


In [58]:
num_stocks = len(market.columns) #we have to set the date as index for this
init_weights = [1 / num_stocks] * num_stocks

In [60]:
annual_market_returns = getPortReturn(init_weights, market)
annual_market_risk = getPortRisk(init_weights, market)
print("Annual market returns:", annual_market_returns)
print("Annual market risk:", annual_market_risk)

Annual market returns: 0.14021749601530095
Annual market risk: 0.2125507152484269


## Question 1.
Optimise your portfolio weights so that the expected return on your 10-15 asset portfolio is equal to the annualised expected return on the market portfolio.

What is the risk of your portfolio now? How does it compare to the market portfolio?

How does it compare to the performance using the initialised weights?

In [61]:
target_return = annual_market_returns

num_stocks = len(portfolio.columns)
init_weights = [1 / num_stocks] * num_stocks
bounds = tuple((0,1) for i in range(num_stocks))
constraints = ({'type' : 'eq',
                'fun' : lambda w : np.sum(w) - 1},
              {'type' : 'eq',
               'fun' : lambda w : w.dot(returns.mean()) * 250 - target_return})

In [62]:
new_weights = minimize(fun=getPortReturn,
                  x0=init_weights,
                  args=(portfolio,),
                  bounds=bounds,
                  constraints=constraints)

In [63]:
print(new_weights)

     message: Positive directional derivative for linesearch
     success: False
      status: 8
         fun: 0.20405812661869388
           x: [ 2.948e-18  1.633e-12  3.219e-14  5.115e-18  3.279e-18
                9.197e-19  2.867e-14  1.077e-14  8.968e-19  1.000e+00]
         nit: 70
         jac: [ 2.955e-01  2.308e-01  5.073e-01  2.566e-01  4.157e-01
                3.062e-01  2.470e-01  7.625e-01  7.494e-01  2.041e-01]
        nfev: 1334
        njev: 66
 multipliers: [ 3.615e+04 -1.566e+05]


In [65]:
print("New annual returns portfolio:",
      getPortReturn(new_weights['x'], portfolio))
print("New annual risk portfolio:",
      getPortRisk(new_weights['x'], portfolio))

New annual returns portfolio: 0.20405812661869388
New annual risk portfolio: 0.2248431184424053


In [67]:
portfolio.pct_change(1).dropna().mean() * 250

Ticker
AAPL    0.295528
AMZN    0.230812
AVGO    0.507285
GOOG    0.256580
LLY     0.415744
META    0.306230
MSFT    0.246955
NVDA    0.762467
TSLA    0.749352
WMT     0.204058
dtype: float64

In [ ]:
#portfolio expected returns can't be less than the market portfolio

## Question 2
Optimise your portfolio weights so that the risk of your 10-15 asset portfolio is equal to the annualised risk of the market portfolio.

What is the expected return on your portfolio now? How does it compare to the market portfolio?

In [72]:
target_risk = annual_market_risk

bounds = tuple((0,1) for i in range(num_stocks))
constraints = ({'type' : 'eq',
        'fun' : lambda w : np.sum(w) - 1},
        {'type' : 'eq',
        'fun' : lambda w : getPortRisk(w, portfolio) - target_risk})

In [73]:
new_risks = minimize(fun=getPortRisk,
                  x0=init_weights,
                  args=(portfolio,),
                  bounds=bounds,
                  constraints=constraints)

In [74]:
print(new_risks)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 0.21255075295449866
           x: [ 1.100e-01  8.746e-02  5.402e-02  1.103e-01  2.224e-01
                2.726e-02  1.023e-01  4.066e-20  4.209e-06  2.864e-01]
         nit: 5
         jac: [ 2.458e-01  2.539e-01  2.641e-01  2.421e-01  2.122e-01
                2.850e-01  2.528e-01  3.483e-01  2.774e-01  1.451e-01]
        nfev: 55
        njev: 5
 multipliers: [ 4.021e-09  1.000e+00]


In [75]:
print("New annual returns portfolio:",
      getPortReturn(new_risks['x'], portfolio))
print("New annual risk portfolio:",
      getPortRisk(new_risks['x'], portfolio))

New annual returns portfolio: 0.2928726663594336
New annual risk portfolio: 0.21255075295449866


## Question 3
Optimise your portfolio weights so that the risk of your portfolio is minimised.

What is the expected return on your portfolio now? How does it compare to the market portfolio?

In [80]:
bounds = tuple((0,1) for i in range(num_stocks))
constraints = ({'type' : 'eq',
        'fun' : lambda w : np.sum(w) - 1})

In [81]:
min_risk = minimize(fun=getPortRisk,
                  x0=init_weights,
                  args=(portfolio,),
                  bounds=bounds,
                  constraints=constraints)

In [82]:
print(min_risk)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 0.19618346753998983
           x: [ 5.090e-02  4.285e-02  2.196e-02  1.229e-01  1.838e-01
                0.000e+00  9.012e-18  0.000e+00  1.447e-17  5.776e-01]
         nit: 10
         jac: [ 1.966e-01  1.967e-01  1.961e-01  1.959e-01  1.963e-01
                2.087e-01  1.991e-01  2.622e-01  2.019e-01  1.961e-01]
        nfev: 110
        njev: 10
 multipliers: [ 1.962e-01]


In [84]:
print("Min risk annual returns portfolio:",
      getPortReturn(min_risk['x'], portfolio))
print("Min risk annual risk portfolio:",
      getPortRisk(min_risk['x'], portfolio))

Min risk annual returns portfolio: 0.2618806297982317
Min risk annual risk portfolio: 0.19618346753998983


## Question 4
Based on the results from your optimisations, would you invest in your portfolio, or would you invest in the market portfolio instead?

Why?

I'd rather invest in my portfolio since the expected returns are higher and the risk is lower when the weights are assigned correctly